<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.4-advanced-retrieval/notebooks/exercises-6.4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.4 — Advanced Retrieval for RAG
Netsetos GenAI Course v2.0 | Module 6

Hybrid search, reranking, query transforms, Self-RAG/CRAG, contextual retrieval, evaluation


In [ ]:
# pip install rank-bm25 FlagEmbedding flashrank ragas deepeval -q
print('Ready for advanced retrieval')


## Ex 1: Hybrid Search (BM25 + Dense + RRF)


In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

corpus = [
    'Error ERR_SSL_PROTOCOL_ERROR on Chrome 120',
    'SSL certificate validation failed during HTTPS',
    'Python pip install fails with SSL error',
    'Machine learning requires GPU for training',
    'The CrowdStrike outage affected 8.5M Windows devices',
]
tokenized = [d.lower().split() for d in corpus]
bm25 = BM25Okapi(tokenized)

def rrf_merge(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1/(k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

query = 'ERR_SSL_PROTOCOL_ERROR'
bm25_scores = bm25.get_scores(query.lower().split())
sparse_ranking = list(np.argsort(-bm25_scores))

# Dense ranking would come from embedding similarity
# For demo, assume dense misses exact error code
dense_ranking = [1, 2, 0, 3, 4]  # SSL-related but not exact match

hybrid = rrf_merge([sparse_ranking, dense_ranking])
print('BM25 top:', corpus[sparse_ranking[0]][:50])
print('Dense top:', corpus[dense_ranking[0]][:50])
print('Hybrid RRF:', [(corpus[d][:40], f'{s:.4f}') for d,s in hybrid[:3]])


## Ex 2: Cross-Encoder Reranking


In [ ]:
# BGE-reranker-v2-m3 (free, 8ms/pair, 18 languages)
# from FlagEmbedding import FlagReranker
# reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)
# scores = reranker.compute_score([
#     ['what is SSL?', 'SSL certificate validation failed'],
#     ['what is SSL?', 'Machine learning requires GPU'],
# ], normalize=True)

# FlashRank (4MB, CPU, <60ms/100 docs)
# from flashrank import Ranker, RerankRequest
# ranker = Ranker(model_name='ms-marco-MiniLM-L-12-v2')
# results = ranker.rerank(RerankRequest(query='your query', passages=data))

print('Reranker Comparison:')
for name, params, lat, cost in [
    ('FlashRank','4MB','<60ms/100','Free'),
    ('BGE-reranker-v2-m3','568M','8ms/pair','Free'),
    ('Jina Reranker v2','278M','15x faster','$0.02/1M tok'),
    ('Cohere Rerank 3.5','Proprietary','~392ms','$2/1K'),
    ('ColBERTv2','~110M','57ms pipeline','Free')]:
    print(f'  {name:25s} | {params:12s} | {lat:15s} | {cost}')


## Ex 3: Query Transformations


In [ ]:
print('=== Step-Back Prompting ===')
original = 'What is the refresh rate of iPhone 13 Pro Max screen?'
step_back = 'What are the technical specifications of the iPhone 13 Pro Max?'
print(f'Original: {original}')
print(f'Step-back: {step_back}')
print('Result: broader retrieval catches the specific answer')
print()

print('=== Query Decomposition ===')
complex_q = 'Compare revenue growth of Apple and Google in Q3 2024'
sub_qs = ['What was Apple revenue growth in Q3 2024?',
          'What was Google revenue growth in Q3 2024?']
print(f'Complex: {complex_q}')
for i, sq in enumerate(sub_qs):
    print(f'  Sub-query {i+1}: {sq}')
print('Result: each sub-query retrieves independently, merge with RRF')
print()

print('=== Query Routing ===')
routes = {
    'What is Python?': 'direct_llm (simple recall)',
    'Latest iPhone price?': 'web_search (current info)',
    'What does our refund policy say?': 'vector_store (private docs)',
}
for q, route in routes.items():
    print(f'  "{q}" → {route}')


## Ex 4: CRAG (Corrective RAG)


In [ ]:
def grade_retrieval(query, documents, threshold=0.3):
    '''Simulate CRAG retrieval grading'''
    # In production: use T5-based evaluator or LLM
    relevance_scores = [0.8, 0.2, 0.1]  # Simulated
    max_score = max(relevance_scores)
    
    if max_score > 0.7:
        return 'correct', documents[:1]
    elif max_score < threshold:
        return 'incorrect', []  # Fall back to web search
    else:
        return 'ambiguous', documents[:1]  # Combine with web search

action, docs = grade_retrieval('What is CRAG?', ['doc1', 'doc2', 'doc3'])
print(f'Action: {action}')
print(f'Docs to use: {docs}')
print()
print('CRAG Pipeline:')
print('  1. Retrieve documents')
print('  2. Grade relevance (correct/incorrect/ambiguous)')
print('  3. Correct → refine + generate')
print('  4. Incorrect → web search fallback')
print('  5. Ambiguous → combine refined docs + web search')
print('  Impact: +26.7pp accuracy on well-retrieved queries')


## Ex 5: Contextual Retrieval (Anthropic)


In [ ]:
context_prompt = '''<document>
{whole_document}
</document>
Here is the chunk we want to situate:
<chunk>
{chunk}
</chunk>
Please give a short succinct context to situate this chunk
within the overall document for improving search retrieval.
Answer only with the succinct context and nothing else.'''

# Example
chunk = 'The company revenue grew by 3% over the previous quarter.'
generated_context = 'This chunk is from ACME corp Q2 2023 SEC filing; previous quarter revenue was $314M.'
contextualized = f'{generated_context} {chunk}'

print(f'Original chunk: {chunk}')
print(f'Contextualized: {contextualized}')
print()
print('Impact Stack:')
print('  Contextual embeddings only:     -35% failures')
print('  + Contextual BM25:              -49% failures')
print('  + Reranking:                    -67% failures')
print(f'  Cost: $1.02/M document tokens (one-time, with prompt caching)')


## Ex 6: RAGAS Evaluation


In [ ]:
print('RAGAS Metrics (target > 0.8):')
metrics = [
    ('Faithfulness', 'claims supported by context / total claims'),
    ('Answer Relevancy', 'reverse Q generation → similarity to original Q'),
    ('Context Precision', 'relevant chunks ranked higher than irrelevant'),
    ('Context Recall', 'context contains all info for ideal answer (needs GT)'),
]
for name, desc in metrics:
    print(f'  {name:25s}: {desc}')

print()
print('Retrieval Metrics:')
for name, use in [('MRR','Single correct answer'),
    ('NDCG@K','Graded relevance (MTEB default)'),
    ('MAP','Multiple relevant, binary'),
    ('Recall@K','Completeness (legal)'),
    ('Precision@K','Noise control')]:
    print(f'  {name:15s}: {use}')

print()
print('Production Targets:')
print('  Faithfulness > 0.8, Context Precision > 0.8')
print('  Answer Relevancy > 0.75, MRR > 0.8')


## Ex 7: Query Router


In [ ]:
from enum import Enum

class QueryRoute(Enum):
    VECTOR_STORE = 'vector_store'
    WEB_SEARCH = 'web_search'
    DIRECT_LLM = 'direct_llm'
    KNOWLEDGE_GRAPH = 'knowledge_graph'

def simple_router(query: str) -> QueryRoute:
    '''Rule-based router (production: use LLM structured output)'''
    q = query.lower()
    if any(w in q for w in ['latest', 'today', 'current', 'news', '2026']):
        return QueryRoute.WEB_SEARCH
    if any(w in q for w in ['what is', 'define', 'explain']) and len(q.split()) < 8:
        return QueryRoute.DIRECT_LLM
    if any(w in q for w in ['relationship', 'connected', 'related to']):
        return QueryRoute.KNOWLEDGE_GRAPH
    return QueryRoute.VECTOR_STORE

test_queries = [
    'What is Python?',
    'Latest iPhone 16 price in India',
    'What does our refund policy say about electronics?',
    'How is revenue connected to customer churn?',
]
for q in test_queries:
    route = simple_router(q)
    print(f'  "{q}" → {route.value}')


## Ex 8: Cross-Lingual Hindi Retrieval


In [ ]:
import unicodedata

def dual_retrieval_pipeline(query_hindi, english_corpus, hindi_corpus):
    '''Dual retrieval for Hindi-English RAG'''
    # Branch 1: Dense retrieval on Hindi corpus
    # hindi_results = dense_search(query_hindi, hindi_corpus)
    
    # Branch 2: Translate → BM25 on English corpus
    # query_english = translate(query_hindi, 'hi', 'en')
    # english_results = bm25_search(query_english, english_corpus)
    
    # Merge via RRF
    # hybrid = rrf_merge([hindi_results, english_results])
    
    # Rerank with multilingual cross-encoder
    # final = rerank(hybrid, query_hindi, model='bge-reranker-v2-m3')
    
    return 'Pipeline: Hindi dense + English BM25 + RRF + multilingual rerank'

def normalize_hindi(text):
    text = unicodedata.normalize('NFC', text)
    for zw in ['\u200b','\u200c','\u200d','\ufeff']:
        text = text.replace(zw, '')
    return ' '.join(text.split())

hindi = 'मशीन\u200bलर्निंग क्या है?'
print(f'Before: {repr(hindi)}')
print(f'After:  {repr(normalize_hindi(hindi))}')
print()
print('Key: normalize BOTH at index time AND query time')
print('Evaluate with ChrF (character F-score, not BLEU)')
print('Cross-lingual accuracy: typically 60-80% of monolingual')


---
## Recap
Advanced retrieval stack: hybrid search (+26% NDCG, non-negotiable) → reranking (+15-40% accuracy, BGE-reranker-v2-m3 for self-hosted) → query routing (highest single ROI, +18-25%, <1ms) → contextual retrieval (-49% failures at $1.02/M tokens) → CRAG for unreliable retrieval (+26.7pp, plug-and-play). Query transforms: step-back for narrow queries (+27%), decomposition for multi-hop (+52%), HyDE for semantic gap. Advanced architectures: Self-RAG (reflection tokens, needs fine-tuning), CRAG (plug-and-play correction), Adaptive RAG (complexity routing), Agentic RAG (LangGraph, cap 3 cycles), GraphRAG (entity relationships). Evaluate: RAGAS faithfulness >0.8 + DeepEval CI/CD + golden test sets. India: dual retrieval (Hindi dense + English BM25 + RRF) + ChrF evaluation.
